# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rimlazrek1/flyrank-internship/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why
*Name your lane — or say 'freestyle' and describe your own question. One short paragraph: why this one?*

**Lane 4 — CTR / Engagement Opportunity Scoring.**

I am not doing the starter refresh problem (“which pages are declining?” using `trend_direction`).

I am asking: **which pages get shown in Google but get fewer clicks than pages with a similar Google rank?**

In the data that means:
- look at `ctr` (clicks ÷ impressions, stored as a percentage like `0.35` = 0.35%)
- compare it within the same `position_tier` (from `avg_position`)
- only trust pages with enough `impressions_90d` so `ctr` is not random noise
- also use `engagement_rate` later if we care about on-page engagement

Why this lane: reviewers already know bad Google rank → lower `ctr`. What they need later is a **ranked list** of pages where `ctr` looks weak **for that `position_tier`**, so they know what to open first. That is decision-support, not “predict Google.”

- One row = one page (`content_id`)
- Later output = ranked CTR/engagement review list

## 2. The question: decision, action, cost of a wrong call
*What decision does your work improve? Who acts on it? What does a wrong recommendation cost?*

**Decision:** Which pages should a reviewer check first because `ctr` (and maybe `engagement_rate`) looks too low for that page’s `avg_position` / `position_tier`, even though `impressions_90d` is high enough to care?

**Who acts:** An editor or SEO person. They open the page, check the Google snippet and the content, then maybe rewrite title/meta, improve the content match, improve engagement, or just monitor.

**Cost of a wrong recommendation (any scoring rule or later model):**
- **False alarm:** we put a healthy page high on our review list → wasted time.
- **Miss:** a real weak-`ctr` page is too far down **our** list → we don’t review it, so we may miss easy click gains.

**Why not only “if `ctr` is low, fix it”?**  
`ctr` alone is not enough:
- Expected `ctr` changes a lot by `position_tier` (top ranks naturally get more clicks).
- If `impressions_90d` is tiny, `ctr` jumps around by luck.

So we need: compare `ctr` to peers in the same `position_tier`, and use a minimum `impressions_90d`. That is why a scored ranked list is better than a simple red-flag column.

**Short frame:** For reviewers choosing which pages to inspect first, we will later build a score from columns like `ctr`, `avg_position`, `position_tier`, `impressions_90d`, and `engagement_rate`, and measure success with something like Precision@K on a holdout. Wrong calls waste time or miss weak-`ctr` pages. We only claim **observed / directional / decision-support** results — not that a fix caused more clicks, and not that we predicted Google’s algorithm.

## 3. Quick look at the data (2-3 real numbers)
*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

Using `data/raw/content_refresh_anonymized.csv`. Reminder: `ctr = 0.35` means **0.35%**, not 35%.

Three simple checks:

1. **`ctr` by `position_tier`**  
   Keep pages with `impressions_90d >= 100` and `avg_position > 0` (`0` means “no position data”).  
   Mean `ctr` is about **0.33–0.36** on `page_1` / `top_3`, but only about **0.06** on `deep`.  
   So “low `ctr`” without `position_tier` is misleading.

2. **Link between `avg_position` and `ctr`**  
   On those same pages, correlation ≈ **-0.24**: higher `avg_position` (worse Google rank) tends to go with lower `ctr`. Directional only — not proof of cause.

3. **How many weak peers?**  
   Keep pages with `impressions_90d >= 500` and `avg_position` between 1 and 20 → **12,023** pages.  
   About **49%** of them have `ctr` **below the median `ctr` of their own `position_tier`**.  
   So many visible pages look weak next to similar-rank peers — enough that building a ranked review list later is useful.

In [2]:
import os
import pandas as pd

# Find repo root (notebook may start in work/notebooks/)
while not os.path.isdir("data/raw") and os.getcwd() != os.path.dirname(os.getcwd()):
    os.chdir("..")
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found"

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# Need enough impressions_90d (less noisy ctr) and a real avg_position (0 = no data)
visible = df[(df["impressions_90d"] >= 100) & (df["avg_position"] > 0)].copy()

print(f"Rows in CSV: {len(df):,}")
print(f"Keep impressions_90d>=100 and avg_position>0: {len(visible):,}\n")

print("Mean of column ctr, grouped by position_tier:")
print(
    visible.groupby("position_tier")["ctr"]
    .mean()
    .sort_values(ascending=False)
    .round(3)
    .to_string()
)

corr = visible["avg_position"].corr(visible["ctr"])
print(f"\nCorrelation between avg_position and ctr: {corr:.3f}")

# Stronger volume floor + only stronger Google ranks (avg_position 1..20)
cand = df[
    (df["impressions_90d"] >= 500)
    & (df["avg_position"] > 0)
    & (df["avg_position"] <= 20)
].copy()
tier_median_ctr = cand.groupby("position_tier")["ctr"].transform("median")
below = cand[cand["ctr"] < tier_median_ctr]
print(f"\nRows with impressions_90d>=500 and avg_position in 1..20: {len(cand):,}")
print(
    f"Rows where ctr < median ctr inside same position_tier: {len(below):,} "
    f"({100 * len(below) / len(cand):.1f}%)"
)

Rows in CSV: 30,000
Keep impressions_90d>=100 and avg_position>0: 22,006

Mean of column ctr, grouped by position_tier:
position_tier
page_1      0.355
top_3       0.334
striking    0.256
page_3_5    0.142
deep        0.055

Correlation between avg_position and ctr: -0.239

Rows with impressions_90d>=500 and avg_position in 1..20: 12,023
Rows where ctr < median ctr inside same position_tier: 5,888 (49.0%)


## 4. Careful words: what I can and can't claim
*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*

**I can say:**
- **Observed:** we measured columns like `ctr`, `impressions_90d`, `avg_position`, `position_tier`, and `engagement_rate` on anonymized pages (`content_id` / `client_id` only).
- **Directional:** pages with weak `ctr` for their `position_tier` (and enough `impressions_90d`) are good candidates to review first.
- **Decision-support:** a ranked list can help humans spend time better; it does not auto-change the site.

**I will not say:**
- Low `ctr` always means a bad title/meta (other things can explain it).
- Editing a page **caused** more clicks, unless we later run a real experiment.
- We “predicted Google’s algorithm.”
- We recovered a product flag like `needs_ctr_fix` (that flag is not in our data on purpose).
- We used `trend_direction` or `trend_pct` as features if the label comes from them (that is leakage).
- Anything with real client names, URLs, titles, or raw queries.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.